# Solutions 3 — Security, Testing, and Measurement

Six worked examples demonstrating validation, secret hygiene, property-style tests, profiling, and integration isolation.

**Level:** Advanced · **Study mode:** Worked reference examples · **Prerequisites:** Relevant lessons and exercise workbook.

These solutions show one defensible approach each; multiple implementations may be correct. Review the contracts, test boundaries, and discuss design trade-offs before comparing line-by-line.

## 1. Bounded JSON parsing

Reject oversized data before parsing, then validate structure after parsing. Real systems should additionally enforce depth and resource limits appropriate to the threat model.

In [ ]:
import json

def parse_payload(text, max_length=1000):
    if not isinstance(text, str) or len(text) > max_length:
        raise ValueError("invalid payload size")
    value = json.loads(text)
    if not isinstance(value, dict) or not isinstance(value.get("items"), list):
        raise ValueError("items array required")
    return value["items"]

assert parse_payload('{"items": [1, 2]}') == [1, 2]
try:
    parse_payload("x" * 2000)
except ValueError:
    print("Oversized input rejected")

**Extension:** Which limits would you enforce before a large upload reaches this function?

## 2. Secret-safe comparison

Use a secure token generator and constant-time comparison where appropriate; never print the token contents. This is not a replacement for password hashing.

In [ ]:
import secrets
import hmac
token = secrets.token_hex(16)
assert len(token) == 32
assert hmac.compare_digest(token, token)
print("Token shape and comparison validated")

**Extension:** Why are token rotation and storage controls still required?

## 3. Seeded property-style tests

Generate reproducible test cases and check a useful invariant. This does not prove correctness but samples more cases than a single example.

In [ ]:
from collections import Counter
import random
rng = random.Random(42)
for _ in range(100):
    values = [rng.randint(-10, 10) for _ in range(rng.randint(0, 30))]
    sorted_values = sorted(values)
    assert sorted_values == sorted(sorted_values)
    assert Counter(sorted_values) == Counter(values)
print("100 deterministic examples passed")

**Extension:** How would a property-testing library shrink a failing input?

## 4. Mock service boundary

A Mock avoids network access while preserving the behavior of the system under test. Test the observable contract rather than implementation details.

In [ ]:
from unittest.mock import Mock

def notify(sender, message):
    if not message:
        raise ValueError("message is required")
    return sender.send(message)

fake = Mock()
fake.send.return_value = "queued"
assert notify(fake, "ready") == "queued"
fake.send.assert_called_once_with("ready")

**Extension:** What failures would a test stub simulate beyond successful delivery?

## 5. Database rollback

SQLite constraints can reject an invalid transaction. Check that a previously committed baseline remains after a failed unit of work.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER CHECK(balance >= 0))")
    db.execute("INSERT INTO accounts VALUES (1, 100)")
    db.commit()
    try:
        with db:
            db.execute("UPDATE accounts SET balance=-20 WHERE id=1")
    except sqlite3.IntegrityError:
        pass
    assert db.execute("SELECT balance FROM accounts WHERE id=1").fetchone()[0] == 100
finally:
    db.close()

**Extension:** Why should transaction tests use actual SQLite instead of mocking every execute call?

## 6. Repeatable timing

Timeit gives environmental observations. Make the benchmark compare equivalent work and discuss construction costs and different lookup distributions.

In [ ]:
import timeit
data = list(range(3000))
fast_set = set(data)
list_seconds = timeit.timeit(lambda: 2999 in data, number=50)
set_seconds = timeit.timeit(lambda: 2999 in fast_set, number=50)
assert 2999 in data and 2999 in fast_set
print(f"list={list_seconds:.6f}s set={set_seconds:.6f}s")

**Extension:** Why does one measurement not demonstrate a general constant speedup?

## Recap

- Reference solutions illustrate testing and contracts rather than exhaustive answers.
- Explain one alternative design for each pattern.
- Re-run every code cell in a fresh kernel before copying into a project.

**Next:** Work independently on the unsolved questions in the corresponding workbook.